<a href="https://colab.research.google.com/github/SantiagoAmaya99/Crack-Detector-Using-Deep-Learning-Models/blob/main/Detecci%C3%B3n_de_grietas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Para implementar este proyecto en Google Colab utilizando aprendizaje débilmente supervisado, utilizaremos **TensorFlow/Keras** y el algoritmo **Grad-CAM** (Gradient-weighted Class Activation Mapping). Este enfoque entrenará una red para clasificar si hay una grieta o no, y luego usará los gradientes de la última capa convolucional para generar un "mapa de calor" que localice la grieta sin haber usado coordenadas en el entrenamiento.

A continuación, tienes la estructura del código dividida en las celdas que debes ejecutar en tu notebook.

**Celda 1: Configuración del entorno y montaje de Google Drive**

In [3]:
from google.colab import drive
drive.mount('/content/drive')
%cd '/content/drive/MyDrive/IA 2/Dataset_IA2'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/.shortcut-targets-by-id/1sSnlg7BIrw5okzzfvdml8bu7TABieBHo/Dataset_IA2


In [7]:
import os
import shutil
from pathlib import Path

# Ruta de origen en tu Google Drive
ruta_origen = Path('/content/drive/MyDrive/IA 2/Dataset_IA2')

# Ruta de destino local temporal en Colab
ruta_destino = Path('/content/dataset')
categoria_grieta_dest = ruta_destino / 'grieta'
categoria_sin_grieta_dest = ruta_destino / 'sin_grieta'

os.makedirs(categoria_grieta_dest, exist_ok=True)
os.makedirs(categoria_sin_grieta_dest, exist_ok=True)

print("Iniciando la consolidación y organización de las imágenes (insensible a mayúsculas, minúsculas y guiones)...")

# Buscar de manera insensible a mayúsculas/minúsculas en el directorio de origen
for elemento in ruta_origen.iterdir():
    if elemento.is_dir() and elemento.name.lower() in ['decks', 'pavements', 'walls']:
        nombre_superficie = elemento.name

        # Buscar carpetas de subcategoría de forma insensible a mayúsculas
        for subelemento in elemento.iterdir():
            if subelemento.is_dir():
                nombre_sub_bajo = subelemento.name.lower().replace('-', '_')

                # Procesar imágenes con grieta
                if nombre_sub_bajo == 'cracked':
                    imagenes = list(subelemento.glob('*'))
                    for img in imagenes:
                        if img.is_file() and img.suffix.lower() in ['.jpg', '.jpeg', '.png']:
                            shutil.copy(img, categoria_grieta_dest / f"{nombre_superficie}_{img.name}")
                    print(f"✓ Copiadas imágenes de {nombre_superficie}/{subelemento.name} -> grieta")

                # Procesar imágenes sin grieta
                elif nombre_sub_bajo == 'non_cracked':
                    imagenes = list(subelemento.glob('*'))
                    for img in imagenes:
                        if img.is_file() and img.suffix.lower() in ['.jpg', '.jpeg', '.png']:
                            shutil.copy(img, categoria_sin_grieta_dest / f"{nombre_superficie}_{img.name}")
                    print(f"✓ Copiadas imágenes de {nombre_superficie}/{subelemento.name} -> sin_grieta")

print("\n¡Consolidación finalizada con éxito!")
print(f"Total imágenes de Grieta (Cracked): {len(os.listdir(categoria_grieta_dest))}")
print(f"Total imágenes Sin Grieta (Non_cracked): {len(os.listdir(categoria_sin_grieta_dest))}")

Iniciando la consolidación y organización de las imágenes (insensible a mayúsculas, minúsculas y guiones)...
✓ Copiadas imágenes de Pavements/Cracked -> grieta
✓ Copiadas imágenes de Pavements/Non-cracked -> sin_grieta
✓ Copiadas imágenes de Decks/Cracked -> grieta
✓ Copiadas imágenes de Decks/Non-cracked -> sin_grieta
✓ Copiadas imágenes de Walls/Cracked -> grieta
✓ Copiadas imágenes de Walls/Non-cracked -> sin_grieta

¡Consolidación finalizada con éxito!
Total imágenes de Grieta (Cracked): 8484
Total imágenes Sin Grieta (Non_cracked): 47608


**Celda 2: Carga y preprocesamiento del Dataset**
Asumiendo que tienes el dataset organizado en carpetas `entrenamiento/grieta` y `entrenamiento/sin_grieta`.

In [8]:
import tensorflow as tf
from tensorflow.keras import layers
import os

# Ajusta esta ruta a la estructura exacta de carpetas que Kaggle haya descomprimido
ruta_dataset = '/content/dataset'
tamano_lote = 32
tamano_imagen = (256, 256)

# Cargar las imágenes directamente desde el disco virtual de Colab
dataset_entrenamiento = tf.keras.utils.image_dataset_from_directory(
    ruta_dataset,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=tamano_imagen,
    batch_size=tamano_lote
)

dataset_validacion = tf.keras.utils.image_dataset_from_directory(
    ruta_dataset,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=tamano_imagen,
    batch_size=tamano_lote
)

# Normalizar los valores de los píxeles
capa_normalizacion = layers.Rescaling(1./255)
dataset_entrenamiento = dataset_entrenamiento.map(lambda x, y: (capa_normalizacion(x), y))
dataset_validacion = dataset_validacion.map(lambda x, y: (capa_normalizacion(x), y))

Found 56092 files belonging to 2 classes.
Using 44874 files for training.
Found 56092 files belonging to 2 classes.
Using 11218 files for validation.


**Celda 3: Definición del Modelo (Arquitectura MobileNetV2 adaptada)**
Usamos MobileNetV2 por ser ligera (ideal para procesamiento rápido). Es crucial incluir `GlobalAveragePooling2D` para que el modelo concentre la información espacial y permita aplicar Grad-CAM.

In [9]:
def construir_modelo():
    # Cargar MobileNetV2 pre-entrenada sin las capas finales
    modelo_base = tf.keras.applications.MobileNetV2(
        input_shape=(256, 256, 3),
        include_top=False,
        weights='imagenet'
    )

    # Congelar el modelo base (opcional, acelera el primer entrenamiento)
    modelo_base.trainable = False

    # Añadir la cabeza de clasificación débilmente supervisada
    entradas = tf.keras.Input(shape=(256, 256, 3))
    x = modelo_base(entradas, training=False)
    x = layers.GlobalAveragePooling2D()(x) # CRÍTICO PARA LOCALIZACIÓN
    x = layers.Dropout(0.2)(x)
    salidas = layers.Dense(1, activation='sigmoid')(x) # Clasificación binaria (Grieta / No Grieta)

    modelo = tf.keras.Model(entradas, salidas)

    modelo.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
    return modelo, modelo_base

modelo_grietas, modelo_base = construir_modelo()
modelo_grietas.summary()

/tmp/ipykernel_7664/900390052.py:3: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  modelo_base = tf.keras.applications.MobileNetV2(


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 256, 256, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 8, 8, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │         1,281 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,259,265 (8.62 MB)

 Trainable params: 1,281 (5.00 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

**Celda 4: Entrenamiento del clasificador**

In [10]:
epocas = 10
historial = modelo_grietas.fit(
    dataset_entrenamiento,
    validation_data=dataset_validacion,
    epochs=epocas
)

Epoch 1/10
1403/1403 ━━━━━━━━━━━━━━━━━━━━ 3451s 2s/step - accuracy: 0.8909 - loss: 0.3018 - val_accuracy: 0.9000 - val_loss: 0.2797
Epoch 2/10
1403/1403 ━━━━━━━━━━━━━━━━━━━━ 3354s 2s/step - accuracy: 0.9055 - loss: 0.2692 - val_accuracy: 0.9061 - val_loss: 0.2670
Epoch 3/10
1403/1403 ━━━━━━━━━━━━━━━━━━━━ 3321s 2s/step - accuracy: 0.9075 - loss: 0.2645 - val_accuracy: 0.9075 - val_loss: 0.2638
Epoch 4/10
1403/1403 ━━━━━━━━━━━━━━━━━━━━ 3325s 2s/step - accuracy: 0.9088 - loss: 0.2599 - val_accuracy: 0.9083 - val_loss: 0.2660
Epoch 5/10
1403/1403 ━━━━━━━━━━━━━━━━━━━━ 3321s 2s/step - accuracy: 0.9095 - loss: 0.2590 - val_accuracy: 0.9070 - val_loss: 0.2711
Epoch 6/10
1403/1403 ━━━━━━━━━━━━━━━━━━━━ 3332s 2s/step - accuracy: 0.9095 - loss: 0.2594 - val_accuracy: 0.9064 - val_loss: 0.2707
Epoch 7/10
1403/1403 ━━━━━━━━━━━━━━━━━━━━ 3334s 2s/step - accuracy: 0.9103 - loss: 0.2568 - val_accuracy: 0.9087 - val_loss: 0.2633
Epoch 8/10
1403/1403 ━━━━━━━━━━━━━━━━━━━━ 3300s 2s/step - accuracy: 0.9093 -

**Celda 5: Implementación de Grad-CAM (El núcleo del proyecto)**
Esta función calcula qué partes de la imagen hicieron que la red neuronal "decidiera" que había una grieta.

In [11]:
def generar_grad_cam(img_array, modelo, nombre_capa_conv):
    # Crear un modelo que devuelve las activaciones de la última capa convolucional y las predicciones finales
    modelo_gradcam = tf.keras.models.Model(
        [modelo.inputs],
        [modelo.get_layer(nombre_capa_conv).output, modelo.output]
    )

    # Registrar las operaciones con GradientTape
    with tf.GradientTape() as tape:
        salida_conv, predicciones = modelo_gradcam(img_array)
        clase_indice = tf.argmax(predicciones[0])
        salida_clase = predicciones[:, clase_indice]

    # Calcular los gradientes de la clase predicha con respecto a la salida convolucional
    grads = tape.gradient(salida_clase, salida_conv)

    # Promediar los gradientes espacialmente
    pesos_canales = tf.reduce_mean(grads, axis=(0, 1, 2))

    # Multiplicar las salidas de la capa convolucional por los pesos
    salida_conv = salida_conv[0]
    mapa_calor = tf.reduce_sum(tf.multiply(salida_conv, pesos_canales), axis=-1)

    # Aplicar ReLU al mapa de calor (solo nos interesan las influencias positivas)
    mapa_calor = np.maximum(mapa_calor, 0)
    mapa_calor /= np.max(mapa_calor) # Normalizar

    return mapa_calor

**Celda 6: Visualización y Localización (El resultado final)**

In [12]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
import glob

# --- OPCIÓN AUTOMÁTICA O MANUAL ---
# Intentamos usar una imagen de tu dataset consolidado para que funcione inmediatamente
imagenes_disponibles = glob.glob('/content/dataset/grieta/*.jpg') + glob.glob('/content/dataset/grieta/*.png')

if imagenes_disponibles:
    # Tomamos la primera imagen con grieta encontrada en el dataset
    ruta_img_prueba = imagenes_disponibles[0]
    print(f"Usando imagen del dataset: {ruta_img_prueba}")
else:
    # Ruta por si prefieres subir tu propia imagen a Drive
    ruta_img_prueba = '/content/drive/MyDrive/imagen_prueba_grieta.jpg'
    print(f"Buscando imagen personalizada en Drive: {ruta_img_prueba}")

# Cargar y preprocesar la imagen
img = tf.keras.preprocessing.image.load_img(ruta_img_prueba, target_size=(256, 256))
img_array = tf.keras.preprocessing.image.img_to_array(img)
img_array_norm = np.expand_dims(img_array, axis=0) / 255.0

# El nombre de la última capa convolucional de MobileNetV2
nombre_ultima_capa = modelo_base.layers[-1].name

# Obtener el mapa de calor de Grad-CAM
mapa_calor = generar_grad_cam(img_array_norm, modelo_base, nombre_ultima_capa)

# Redimensionar el mapa de calor al tamaño original
mapa_calor_redimensionado = cv2.resize(mapa_calor, (img_array.shape[1], img_array.shape[0]))
mapa_calor_redimensionado = np.uint8(255 * mapa_calor_redimensionado)

# Aplicar colormap (escala de temperatura jet)
mapa_color = cv2.applyColorMap(mapa_calor_redimensionado, cv2.COLORMAP_JET)

# Superponer mapa de calor sobre la imagen original
imagen_superpuesta = cv2.addWeighted(np.uint8(img_array), 0.6, mapa_color, 0.4, 0)

# Mostrar resultados
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.imshow(np.uint8(img_array))
plt.title("Imagen Original (De Prueba)")
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(imagen_superpuesta)
plt.title("Localización con Grad-CAM")
plt.axis('off')
plt.show()

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/imagen_prueba_grieta.jpg'